# 3.9 Object Tracking

The following notebook will: 

1. Load a YOLOv8 model
2. Will run detection on a video
3. Performs simple multi-object ID tracking by matching detection centers frame-to-frame
4. Will create and save an Output that does the following: 
 
    A) Will create and save an annotated .mp4 with bounding boxes, class names, confidences, and track IDs

## Install OpenCV

In [192]:
import sys
!{sys.executable} -m pip install opencv-python

## Install Ultralytics

In [195]:
import sys
!{sys.executable} -m pip install ultralytics

In [197]:
import os
from pathlib import Path
import math
import cv2

from ultralytics import YOLO

## Relative Paths

I will be using relative paths, with the code below. This will allow for easier use on differnt devices.

In [200]:
from pathlib import Path

BASE_DIR = Path.cwd() / "object_tracking"
VIDEO_PATH = BASE_DIR / "videos" / "trucks.mp4"
MODEL_PATH = BASE_DIR / "dnn_model" / "yolov8n.pt"
OUTPUT_PATH = BASE_DIR / "output_trucks_yolov8_tracking.mp4"

print("BASE_DIR:", BASE_DIR)
print("VIDEO_PATH exists:", VIDEO_PATH.exists(), VIDEO_PATH)
print("MODEL_PATH exists:", MODEL_PATH.exists(), MODEL_PATH)

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

BASE_DIR: C:\Users\MasterDanteDev86\Downloads\CAP415-O Computer Vision - Online\W3\object_tracking
VIDEO_PATH exists: True C:\Users\MasterDanteDev86\Downloads\CAP415-O Computer Vision - Online\W3\object_tracking\videos\trucks.mp4
MODEL_PATH exists: True C:\Users\MasterDanteDev86\Downloads\CAP415-O Computer Vision - Online\W3\object_tracking\dnn_model\yolov8n.pt


## Loading YOLOv8 Model

The YOLOv8 will provide detection results directly from the model call, and we can access:

1. Model.names for class name strings
2. Results.boxes.data which includes

    A) [xmin, ymin, xmax, ymax, conf, class_id]

In [203]:
model = YOLO(str(MODEL_PATH))
class_names = model.names

print("Model loaded.")
print("Number of classes:", len(class_names))

Model loaded.
Number of classes: 80


## Helper Functions

Helpers:

1. Will convert YOLO output to clean ints
2. Will compute center points
3. Will draw boxes and labels

In [206]:
def clamp_int(value, low, high):
    if value < low:
        return int(low)
    if value > high:
        return int(high)
    return int(value)

def get_center(xmin, ymin, xmax, ymax):
    cx = (xmin + xmax) / 2.0
    cy = (ymin + ymax) / 2.0
    return (cx, cy)

def distance(p1, p2):
    dx = p1[0] - p2[0]
    dy = p1[1] - p2[1]
    return math.sqrt(dx * dx + dy * dy)

def draw_label(frame, text, x, y):
    # will draw a label with a background rectangle
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.5
    thickness = 1

    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
    # background box
    cv2.rectangle(frame, (x, y - th - baseline - 4), (x + tw + 4, y + 2), (0, 0, 0), -1)
    cv2.putText(frame, text, (x + 2, y - 2), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)

## ID Tracker

The tracker will do centroid matching:

1. Each detection becomes a center point.

2. Will match each new center to the nearest existing track within a distance threshold.

3. If there is no match, then a new ID is created.

4. Tracks that have not been seen in a while will be removed.

In [209]:
class SimpleTracker:
    def __init__(self, dist_threshold=60.0, max_missing=20):
        self.dist_threshold = float(dist_threshold)
        self.max_missing = int(max_missing)

        self.next_id = 1
        self.tracks = {}

    def update(self, detections):
        """
        detections: list of dicts:
          {
            'bbox': (xmin,ymin,xmax,ymax),
            'center': (cx,cy),
            'conf': float,
            'class_id': int
          }
        returns: list of dicts with 'track_id' added
        """
        for tid in self.tracks:
            self.tracks[tid]["missing"] += 1

        used_tracks = set()
        output = []

        for det in detections:
            best_tid = None
            best_dist = None

            for tid, info in self.tracks.items():
                if tid in used_tracks:
                    continue

                d = distance(det["center"], info["center"])
                if best_dist is None or d < best_dist:
                    best_dist = d
                    best_tid = tid

            if best_tid is not None and best_dist is not None and best_dist <= self.dist_threshold:
                used_tracks.add(best_tid)
                self.tracks[best_tid]["center"] = det["center"]
                self.tracks[best_tid]["missing"] = 0
                self.tracks[best_tid]["class_id"] = det["class_id"]

                det_out = dict(det)
                det_out["track_id"] = best_tid
                output.append(det_out)
            else:
                # create a new track
                new_id = self.next_id
                self.next_id += 1

                self.tracks[new_id] = {
                    "center": det["center"],
                    "missing": 0,
                    "class_id": det["class_id"]
                }

                det_out = dict(det)
                det_out["track_id"] = new_id
                output.append(det_out)

        to_delete = []
        for tid, info in self.tracks.items():
            if info["missing"] > self.max_missing:
                to_delete.append(tid)

        for tid in to_delete:
            del self.tracks[tid]

        return output

## Video Processing Settings

Here I will set the following:

1. Confidence threshold
2. Distance threshold for tracking
3. Output video writer settings

In [212]:
CONF_THRESH = 0.30
TRACK_DIST_THRESH = 70.0
MAX_MISSING = 25

tracker = SimpleTracker(dist_threshold=TRACK_DIST_THRESH, max_missing=MAX_MISSING)

## Run Tracking and Saving Output MP4

The code below will:

1. Reads frames from trucks.mp4
2. Runs YOLOv8 detections
3. Converts results to bbox + center points
4. Updates tracker IDs
5. Draws box and "ID: X class conf" on the frame
6. Saves to output_trucks_yolov8_tracking.mp4

In [215]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError("Could not open video.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_PATH), fourcc, fps, (width, height))

frame_index = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_index += 1

    results = model(frame)[0]
    # YOLOv8 returns detections in the format:
    # [xmin, ymin, xmax, ymax, confidence, class_id]
    data = results.boxes.data.tolist()

    detections = []
    for row in data:
        xmin, ymin, xmax, ymax = row[0], row[1], row[2], row[3]
        conf = float(row[4])
        class_id = int(row[5])
        # Filter weak detections and convert box coords to safe ints
        if conf < CONF_THRESH:
            continue

        xi1 = clamp_int(xmin, 0, width - 1)
        yi1 = clamp_int(ymin, 0, height - 1)
        xi2 = clamp_int(xmax, 0, width - 1)
        yi2 = clamp_int(ymax, 0, height - 1)

        cx, cy = get_center(xi1, yi1, xi2, yi2)

        detections.append({
            "bbox": (xi1, yi1, xi2, yi2),
            "center": (cx, cy),
            "conf": conf,
            "class_id": class_id
        })

    tracked = tracker.update(detections)

    for det in tracked:
        (x1, y1, x2, y2) = det["bbox"]
        tid = det["track_id"]
        conf = det["conf"]
        class_id = det["class_id"]

        if isinstance(class_names, dict):
            cname = class_names.get(class_id, str(class_id))
        else:
            cname = class_names[class_id] if class_id < len(class_names) else str(class_id)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (40, 220, 40), 2)
        # Draw bounding box and a label showing: ID, class name, and confidence
        label = f"ID:{tid} {cname} {conf:.2f}"
        draw_label(frame, label, x1, y1)

    writer.write(frame)

cap.release()
writer.release()

print("Done.")
print("Saved:", OUTPUT_PATH)


0: 384x640 1 car, 2 trucks, 47.0ms
Speed: 2.7ms preprocess, 47.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 42.8ms
Speed: 2.3ms preprocess, 42.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 37.8ms
Speed: 1.5ms preprocess, 37.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 34.8ms
Speed: 1.1ms preprocess, 34.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 39.3ms
Speed: 2.0ms preprocess, 39.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 35.3ms
Speed: 1.5ms preprocess, 35.3ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 35.3ms
Speed: 1.2ms preprocess, 35.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 2 trucks, 39.3ms
Speed: 1.2ms preprocess, 39.3ms i

## Reflection

I used YOLOv8n to detect vehicles in the trucks.mp4 video, and then applied a simple centroid based tracker to assign IDs across frames. Each detected object is labeled with an ID, the predicted class name, and the confidence score.

In the output video, the trucks and cars are detected correctly, and each object keeps a consistent ID as it moves across the frame. For example, the truck labeled “ID:17” and the car labeled “ID:18” both maintain stable tracking while driving down the highway.

I used a confidence threshold of 0.30 to filter weak detections. The distance threshold of 70 helps match objects between frames without switching IDs too easily. If objects get very close together, there is still a chance IDs may switch, but overall the tracking works consistently.

The final output video shows clear bounding boxes, class labels, confidence scores, and tracking IDs.

IDs stayed stable for most cars/trucks, but when two vehicles overlapped, IDs could switch briefly.

Lowering/raising TRACK_DIST_THRESH changed how often IDs switched.